# DESeq2 Benchmark — Analysis 1: Genes Rescued by Normative Modeling

DESeq2's independent filtering drops genes whose `baseMean` is too low for a reliable
Wald test (`pvalue` is NA in the result table). This notebook checks whether the
Normative Modeling engine still produces a disease-specific flagged signal for those
excluded genes -- across all three sub-models (NBI/ZINBI, logistic, and the separate
RareEventScorer for det_rate < low_det_thr genes, which `Z_disease.npy` does not cover).
Flagged calls are read from `disease_scores_flagged.parquet`, the only artifact where the
rare-event branch is actually scored.

Phenotype: `Liver Cancer (Roskams-Hieter)`

In [ ]:
%load_ext autoreload
%autoreload 2

In [11]:
import sys
sys.path.insert(0, '.')
sys.path.insert(0, '..')
import config
import pandas as pd
from pipeline import data_prep, plots
from pipeline.benchmark import gene_level_compare, rescued_genes

In [ ]:
dd = data_prep.load_disease_filtered()

PHENOTYPES = print(pd.Series(dd.dis_pheno).unique)

for PHENOTYPE in PHENOTYPES : 
    print(PHENOTYPE)
    merged = gene_level_compare(dd, PHENOTYPE)
    print(f"genes compared: {len(merged)}  |  DESeq2-excluded: {merged['excluded'].sum()}")
    rescued = rescued_genes(merged)
    rescued[['symbol', 'ensg', 'baseMean', 'branch', 'det_rate',
            'norm_n_flagged', 'norm_flagged_prop', 'norm_max_abs_z']]

    out_path = config.BENCHMARK_DIR / f"rescued_genes_{PHENOTYPE.replace(' ', '_').replace('/', '_')}.csv"
    rescued.to_csv(out_path, index=False)
    print(f'saved -> {out_path}')
    _ = plots.plot_rescued_genes(merged, rescued, PHENOTYPE) 